# torch.compile 与图捕获

## 学习目标

在支持的 PyTorch 版本上运行 `torch.compile`，比较 eager 与 compiled 模式，并识别动态 Python 控制流导致的 graph break。

## 概念模型

compile 会捕获并优化 Tensor 运算图。首次调用通常有编译开销，性能结论必须在预热后比较；不支持的 Python 行为可能触发 graph break 或回退。

In [ ]:
import torch
from torch import nn

model = nn.Sequential(nn.Linear(8, 16), nn.ReLU(), nn.Linear(16, 2))
inputs = torch.randn(4, 8)
print('torch:', torch.__version__, 'compile available:', hasattr(torch, 'compile'))
eager = model(inputs)
assert eager.shape == (4, 2)

### 实验 1：编译模型并验证输出

**实验目的**：用 `backend='eager'` 验证 `torch.compile` 捕获接口和行为一致性，不以性能为目标。首次调用包含编译开销，必须与 eager 输出做容差比较。

真实加速需使用适合后端、预热并重复测量；模型规模、shape 稳定性和 graph break 都会影响收益。


In [ ]:
if hasattr(torch, 'compile'):
    compiled = torch.compile(model, backend='eager')
    compiled_output = compiled(inputs)
    torch.testing.assert_close(compiled_output, eager)
    print('compiled output:', compiled_output.shape)
else:
    print('torch.compile unavailable; skipped')

### 实验 2：动态控制流的图捕获边界

**实验目的**：展示依赖张量值的 Python `if` 可能导致 graph break、重编译或后端限制。不同输入分支都应单独验证。

`backend='eager'` 更适合诊断捕获行为；生产优化前可用 explain/日志定位 graph break，并避免无界动态 shape 导致编译缓存膨胀。


In [ ]:
class DynamicModel(nn.Module):
    def forward(self, x):
        if x.sum() > 0:
            return x * 2
        return x - 2

dynamic = DynamicModel()
if hasattr(torch, 'compile'):
    dynamic_compiled = torch.compile(dynamic, backend='eager')
    print('dynamic result:', dynamic_compiled(torch.ones(2)))
else:
    print('compile skipped')

## 官方教程补充

**对应官方源文件：** `intermediate_source/torch_compile_tutorial.py`、`intermediate_source/torch_compile_full_example.py`、`recipes_source/torch_logs.py`、`recipes_source/regional_compilation.py`

官方 `torch.compile` 教程把编译看作捕获 Python 执行并生成优化图：首次调用含编译成本，后续匹配 guard 的输入才复用。数据相关 Python 控制流、动态 shape 或副作用可能造成 graph break/recompile；用日志解释原因。性能比较必须预热并包含多个稳态迭代，且始终先比较 eager 与 compiled 的数值和梯度。

**验证练习：** 找到上面源文件中的对应 API，先写出输入、输出和状态变化，再运行本 notebook 的相关实验；如果行为不同，优先检查本地 PyTorch 版本、设备能力和输入契约。

<!-- official-pytorch-supplement-v1 -->

## 检查点

解释首次编译开销、预热、graph break 和输出一致性验证。

## 试一试

用 `torch.profiler` 比较多次 eager/compiled 调用；尝试不同 batch size 并记录是否出现重新编译。

## 常见错误与调试

只测首次调用、把小模型的编译时间当作稳定收益、未验证数值一致性、忽略动态 shape 和 Python 控制流。